In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

zarr_path = '/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr'

matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)

In [2]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

In [3]:
import global_gauges as gg

facade = gg.GaugeDataFacade()
# sites = facade.get_stations_n_days(90)

In [4]:
# processing_status = store.get_processing_status(source='gauge')
# processed_basins = processing_status.index.get_level_values('subbasin')
# to_process = matchups[~matchups.index.isin(processed_basins)]
to_process = matchups

max_batch_size = 64
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        if not row['custom']:
            gauge_df = None
        else:
            # There is not an explicit flag for custom that is not a gauge, so we just try
            # and then catch the error if this comid doesn't exist in our gauge dataset.
            try:
                gauge_df = facade.get_daily_values(comid).droplevel(axis=0, level=0)
                gauge_df = gauge_df[['discharge']]
                gauge_df['sp_discharge'] = gauge_df / row['total_area']
            except ValueError:
                gauge_df = None
                
        subbasin_data[comid] = gauge_df

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')
            subbasin_data = {}
            
    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')

100%|██████████| 643/643 [3:00:58<00:00, 16.89s/it]    


In [ ]:
next(generate_tasks(matchups))

In [ ]:
gauge_df

In [ ]:
df = store.read_dynamic(basin_id, comid, 'gauge', '2020-01-01', '2020-12-31')['gauge']

In [ ]:
subbasin_data[comid]

In [ ]:
gauge_df['sp_discharge'].max()

In [ ]:
processed_basins = store.get_processing_status(source='swot-river')
processed_basins

In [ ]:
processed_basins[processed_basins['has_data']]['basin'].value_counts()

In [ ]:
store.compact()

In [ ]:
matchups['lake_pld_ids'].apply(len).gt(0).sum()